<a href="https://colab.research.google.com/github/prince24-web/Mechine_learning/blob/main/Student_Exam_Pass_Prediction(logistic_regression).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install ucimlrepo

In [10]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
student_performance = fetch_ucirepo(id=320)

# data (as pandas dataframes), making explicit copies to avoid SettingWithCopyWarning
X = student_performance.data.features.copy()
y = student_performance.data.targets.copy()

# metadata
print(student_performance.metadata)

# variable information
print(student_performance.variables)

{'uci_id': 320, 'name': 'Student Performance', 'repository_url': 'https://archive.ics.uci.edu/dataset/320/student+performance', 'data_url': 'https://archive.ics.uci.edu/static/public/320/data.csv', 'abstract': 'Predict student performance in secondary education (high school). ', 'area': 'Social Science', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 649, 'num_features': 30, 'feature_types': ['Integer'], 'demographics': ['Sex', 'Age', 'Other', 'Education Level', 'Occupation'], 'target_col': ['G1', 'G2', 'G3'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2008, 'last_updated': 'Fri Jan 05 2024', 'dataset_doi': '10.24432/C5TG7T', 'creators': ['Paulo Cortez'], 'intro_paper': {'ID': 360, 'type': 'NATIVE', 'title': 'Using data mining to predict secondary school student performance', 'authors': 'P. Cortez, A. M. G. Silva', 'venue': 'Proceedings of 5th Annual Future Business Technolo

In [5]:
print('Features (X) head:')
display(X.head())

Features (X) head:


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,yes,no,no,4,3,4,1,1,3,4
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,yes,yes,no,5,3,3,1,1,3,2
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,yes,yes,no,4,3,2,2,3,3,6
3,GP,F,15,U,GT3,T,4,2,health,services,...,yes,yes,yes,3,2,2,1,1,5,0
4,GP,F,16,U,GT3,T,3,3,other,other,...,yes,no,no,4,3,2,1,2,5,0


In [4]:
print('\nTargets (y) head:')
display(y.head())


Targets (y) head:


,G1,G2,G3
0,0,11,11
1,9,11,11
2,12,13,12
3,14,14,14
4,11,13,13


In [11]:
import numpy as np

# Create 'avg_prev_grades' feature by averaging G1 and G2
X['avg_prev_grades'] = (y['G1'] + y['G2']) / 2
display(y[['G1', 'G2']].head()) # Display G1 and G2 from y
display(X[['avg_prev_grades']].head()) # Display avg_prev_grades from X

,G1,G2
0,0,11
1,9,11
2,12,13
3,14,14
4,11,13


,avg_prev_grades
0,5.5
1,10.0
2,12.5
3,14.0
4,12.0


In [14]:
import numpy as np

# Create 'avg_prev_grades' feature by averaging G1 and G2
X['avg_prev_grades'] = (y['G1'] + y['G2']) / 2

# Convert 'famsup' and 'internet' from 'yes'/'no' to 1/0
X['famsup_numeric'] = X['famsup'].apply(lambda x: 1 if x == 'yes' else 0)
X['internet_numeric'] = X['internet'].apply(lambda x: 1 if x == 'yes' else 0)

print("Features after creation:")
display(X[['avg_prev_grades', 'famsup_numeric', 'internet_numeric']].head())

Features after creation:


,avg_prev_grades,famsup_numeric,internet_numeric
0,5.5,0,0
1,10.0,1,1
2,12.5,0,1
3,14.0,1,1
4,12.0,1,0


### 4. Family Support

The `famsup` column in `X` indicates 'family educational support' ('yes' or 'no'), which is directly relevant. The `famrel` column (family relationship quality, from 1 to 5) could also be a valuable feature representing family support.

For `famsup`, you'll want to convert the 'yes'/'no' values into numerical ones (e.g., 1 for 'yes', 0 for 'no').

### 5. Internet Access

Similar to family support, the `internet` column in `X` tells you if the student has internet access at home ('yes' or 'no'). This needs to be converted to a numerical format for your model.

Here's how you can convert the `famsup` and `internet` columns into numerical features:

In [7]:
# Convert 'famsup' and 'internet' from 'yes'/'no' to 1/0
X['famsup_numeric'] = X['famsup'].apply(lambda x: 1 if x == 'yes' else 0)
X['internet_numeric'] = X['internet'].apply(lambda x: 1 if x == 'yes' else 0)

display(X[['famsup', 'famsup_numeric', 'internet', 'internet_numeric']].head())

/tmp/ipykernel_5061/1890611984.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['famsup_numeric'] = X['famsup'].apply(lambda x: 1 if x == 'yes' else 0)


,famsup,famsup_numeric,internet,internet_numeric
0,no,0,no,0
1,yes,1,yes,1
2,no,0,yes,1
3,yes,1,yes,1
4,yes,1,no,0


### Creating the Target Variable: Pass or Fail

Finally, for your logistic regression model, you need a binary target variable representing 'pass' or 'fail'. The `G3` column in your `y` DataFrame is the final grade. A common passing threshold is 10 out of 20 (assuming the grades are out of 20, which is typical for this dataset).

Let's create a new target column `pass_fail` where 1 means 'pass' (G3 >= 10) and 0 means 'fail' (G3 < 10):

In [20]:
# Create the binary target variable 'pass_fail'
y['pass_fail'] = y['G3'].apply(lambda grade: 1 if grade >= 10 else 0)
display(y[['G3', 'pass_fail']].head())

,G3,pass_fail
0,11,1
1,11,1
2,12,1
3,14,1
4,13,1


Now you have your `X` DataFrame with several processed features and your `y` DataFrame with the new `pass_fail` target. You can select the specific features you want to include in your logistic regression model from `X`.

In [24]:
# Define the list of selected features
selected_features = [
    'studytime',         # Already numerical
    'absences',          # Used as attendance
    'avg_prev_grades',   # Created by averaging G1 and G2
    'famsup_numeric',    # Converted from 'yes'/'no'
    'internet_numeric'   # Converted from 'yes'/'no'
]

# Extract features into X_model
X_model = X[selected_features]

# Extract the target variable into y_model (assuming 'pass_fail' has been created in y)
y_model = y['pass_fail']

print("Selected Features (X_model) head:")
display(X_model.head())

print("\nTarget Variable (y_model) head:")
display(y_model.head())
display(y_model.value_counts())

Selected Features (X_model) head:


,studytime,absences,avg_prev_grades,famsup_numeric,internet_numeric
0,2,4,5.5,0,0
1,2,2,10.0,1,1
2,2,6,12.5,0,1
3,3,0,14.0,1,1
4,2,0,12.0,1,0



Target Variable (y_model) head:


,pass_fail
0,1
1,1
2,1
3,1
4,1


,count
pass_fail,
1,549
0,100


In [33]:
print("Missing values in X_model (features):")
display(X_model.isnull().sum())

Missing values in X_model (features):


,0
studytime,0
absences,0
avg_prev_grades,0
famsup_numeric,0
internet_numeric,0


In [34]:
print("\nMissing values in y_model (target):")
display(y_model.isnull().sum())


Missing values in y_model (target):


np.int64(0)

In [36]:
from sklearn.model_selection import train_test_split
train_test_split(X_model, y_model, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
print(X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape)


(415, 5) (104, 5) (130, 5) (415,) (104,) (130,)


In [37]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model
# You can add parameters like C (regularization strength) or solver if needed
model = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is a good default for small datasets

# Train the model using the training data
model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


With the model trained, the final step is to **evaluate its performance** on the unseen `X_test` data. This will tell us how well our model generalizes to new student data.

Would you like to proceed with evaluating the model's performance?

In [38]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.4f}")
print(f"Model Precision: {precision:.4f}")
print(f"Model Recall: {recall:.4f}")
print(f"Model F1-Score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
display(confusion_matrix(y_test, y_pred))

Model Accuracy: 0.9154
Model Precision: 0.9333
Model Recall: 0.9739
Model F1-Score: 0.9532

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.47      0.56        15
           1       0.93      0.97      0.95       115

    accuracy                           0.92       130
   macro avg       0.82      0.72      0.76       130
weighted avg       0.91      0.92      0.91       130


Confusion Matrix:


array([[  7,   8],
       [  3, 112]])

These metrics provide a comprehensive view of your model's performance.

*   **Accuracy** tells you the proportion of correctly classified instances.
*   **Precision** is about the quality of positive predictions (of all predicted 'pass', how many actually 'pass').
*   **Recall** is about the completeness of positive predictions (of all actual 'pass', how many did we correctly identify).
*   **F1-Score** is the harmonic mean of precision and recall, offering a balance between them.

We have now completed the requested steps:
1.  Prepared features for logistic regression.
2.  Created the target 'pass'/'fail' variable.
3.  Checked for missing values.
4.  Split the data into training and testing sets.
5.  Trained a Logistic Regression model.
6.  Evaluated the model's performance.

Would you like to analyze these results further, perhaps look at feature importance, or try another model?

If the output shows any columns with a count greater than zero, it means there are missing values that you might need to handle (e.g., by imputation or removal) before proceeding with model training. Since all the columns we selected are either numerical or have been explicitly converted, and the original dataset was stated to have no missing values, we expect to see zero missing values.

Now you have `X_model` containing your chosen features and `y_model` as your binary target variable. The next steps would typically involve:

1.  **Splitting the data:** Divide your dataset into training and testing sets.
2.  **Training the model:** Fit a logistic regression model using your training data.
3.  **Evaluating the model:** Assess the model's performance on the test set using metrics like accuracy, precision, recall, or F1-score.

Would you like me to guide you through these next steps for building and evaluating your logistic regression model?

In [40]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
# We use stratify=y_model to maintain the same proportion of pass/fail in both sets
X_train, X_test, y_train, y_test = train_test_split(X_model, y_model, test_size=0.2, random_state=42, stratify=y_model)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nDistribution of 'pass_fail' in original dataset:")
display(y_model.value_counts(normalize=True))

print("\nDistribution of 'pass_fail' in training set:")
display(y_train.value_counts(normalize=True))

print("\nDistribution of 'pass_fail' in test set:")
display(y_test.value_counts(normalize=True))

Shape of X_train: (519, 5)
Shape of X_test: (130, 5)
Shape of y_train: (519,)
Shape of y_test: (130,)

Distribution of 'pass_fail' in original dataset:


,proportion
pass_fail,
1,0.845917
0,0.154083



Distribution of 'pass_fail' in training set:


,proportion
pass_fail,
1,0.845857
0,0.154143



Distribution of 'pass_fail' in test set:


,proportion
pass_fail,
1,0.846154
0,0.153846


Now that your data is split, the next steps are:

1.  **Train the Logistic Regression model:** Fit the model using `X_train` and `y_train`.
2.  **Evaluate the model:** Make predictions on `X_test` and compare them to `y_test` using appropriate metrics.

Would you like to proceed with training the model?